# Reformat SY Export

### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
from typing import Tuple

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm  # Progress bar

import climakitae as ck
from climakitae.explore.standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
    set_profile_metadata,
    _format_based_on_structure,
    _construct_profile_dataframe,
    _create_simple_dataframe,
    _create_single_wl_multi_sim_dataframe,
    _create_multi_wl_single_sim_dataframe,
    _create_multi_wl_multi_sim_dataframe,
     _stack_profile_data
)
from climakitae.explore.typical_meteorological_year import TMY
from climakitae.core.data_interface import (
    get_data_options,
    get_subsetting_options,
    get_data,
)

import warnings
warnings.filterwarnings("ignore")


from unittest.mock import MagicMock, call, patch
import pytest

## Where is the issue?

### Scratch

In [2]:
def compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time_delta)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
    )

    # Eagerly compute all dask data at once (one round-trip to scheduler)
    if hasattr(data.data, "chunks"):
        print("      📥 Loading data into memory...")
        from dask.diagnostics import ProgressBar

        with ProgressBar():
            data = data.compute()

    # Initialize storage for profiles
    profile_data = {}
    #! -start
    profile_data_reshaped = {}
    profile_data_1d = {}
    #! -end

    # Progress tracking
    total_combinations = len(warming_levels) * n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        for wl_idx, wl in enumerate(warming_levels):
            for sim_idx, sim in enumerate(simulations):
                # Get simulation label
                sim_label = _get_simulation_label(sim, sim_idx)

                # Select data for this warming level and simulation combination
                if has_simulation:
                    subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                else:
                    subset_data = data.isel(warming_level=wl_idx)

                # Vectorized quantile computation using numpy
                # Reshape raw values into (n_years, hours_per_year) then compute
                # the quantile across years for each hour-of-year position
                values = subset_data.values
                n_total = len(values)
                usable = (n_total // hours_per_year) * hours_per_year
                year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

                # Compute quantile targets for each of the 8760 hour positions
                quantile_targets = np.nanquantile(
                    year_hour_matrix, q, axis=0
                )  # shape: (8760,)

                # For each hour position, find the actual year whose value is
                # closest to the quantile (avoids interpolation)
                diffs = np.abs(
                    year_hour_matrix - quantile_targets[np.newaxis, :]
                )  # (n_years, 8760)
                closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)
                profile_1d = year_hour_matrix[
                    closest_year_idx, np.arange(hours_per_year)
                ]

                # Reshape to (days_in_year, 24) for the final DataFrame
                profile_reshaped = profile_1d[: days_in_year * hours_per_day].reshape(
                    days_in_year, hours_per_day
                )

                # Store the profile
                key = (f"WL_{wl}", sim_label)
                profile_data[key] = profile_reshaped
                #! -start
                profile_data_reshaped[key] = profile_reshaped
                profile_data_1d[key] = profile_1d
                #! -end

                pbar.update(1)

    # # Create the multi-index DataFrame structure
    # df_profile = _construct_profile_dataframe(
    #     profile_data=profile_data,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -start
    # df_profile_reshaped = _construct_profile_dataframe(
    #     profile_data=profile_data_reshaped,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # df_profile_1d = _construct_profile_dataframe(
    #     profile_data=profile_data_1d,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -end

    # # Determine which formatting function to use based on the structure
    # _format_based_on_structure(df_profile)

    # #! -start
    # _format_based_on_structure(df_profile_reshaped)
    # _format_based_on_structure(df_profile_1d)
    # #! -end

    # # Prepare metadata dictionary
    # metadata = {
    #     "quantile": q,
    #     "method": "8760 analysis - actual data closest to quantile across 30 years",
    #     "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    # }

    # # Add original data attributes if available
    # if hasattr(data, "attrs"):
    #     if "units" in data.attrs:
    #         metadata["units"] = data.attrs["units"]
    #     if "extended_description" in data.attrs:
    #         metadata["extended_description"] = data.attrs["extended_description"]
    #     if "variable_id" in data.attrs:
    #         metadata["variable_name"] = data.attrs["variable_id"]
    #     elif hasattr(data, "name") and data.name:
    #         metadata["variable_name"] = data.name

    # # Set all metadata using the helper function
    # set_profile_metadata(df_profile, metadata)

    # #! -start
    # set_profile_metadata(df_profile_reshaped, metadata)
    # set_profile_metadata(df_profile_1d, metadata)
    # #! -end

    # print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    # print(
    #     f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    # )
    # if hasattr(data, "attrs") and "units" in data.attrs:
    #     print(f"         Units: {data.attrs['units']}")

    return profile_data_reshaped, profile_data_1d

#### Multi wl, single sim

In [46]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5, 2.0]
simulations = ["sim1"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

sample_data_multi_wl = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

profile_data_reshaped_multi_wl, profile_data_1d_multi_wl = compute_profile(sample_data_multi_wl, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

In [47]:
# Create helper function to extract meaningful simulation labels
def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
    """Extract meaningful simulation label from simulation identifier."""
    if sim is None:
        return f"Sim_{sim_idx+1}"

    sim_str = str(sim)
    if "WRF_" in sim_str:
        # Parse simulation name format: WRF_GCM_params_scenario
        # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
        parts = sim_str.split("_")
        if len(parts) >= 4:
            gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
            params = parts[2]  # e.g., r11i1p1f1
            scenario = parts[3]  # e.g., historical+ssp245

            # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
            if "ssp" in scenario:
                ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                ssp = f"ssp{ssp_part}"
            else:
                ssp = "hist"  # fallback for historical-only

            return f"{gcm}-{params}-{ssp}"
        elif len(parts) >= 2:
            # Fallback for shorter format
            return f"{parts[1]}-{sim_idx+1}"
        else:
            return f"Sim_{sim_idx+1}"
    else:
        # Ensure uniqueness by adding index for non-WRF format
        base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
        return f"{base_name}-{sim_idx+1}"

In [48]:
data = sample_data_multi_wl
days_in_year: int = 365
q = 0.5

warming_levels = data.warming_level.values
# Check for simulation dimension
has_simulation = "simulation" in data.dims
if has_simulation:
    n_simulations = len(data.simulation)
    simulations = data.simulation.values
else:
    n_simulations = 1
    simulations = [None]

profile_data = profile_data_1d_multi_wl
sim_label_func=_get_simulation_label,

hours_per_day = 24
hours_per_year = 8760

In [50]:
simulations = ['sim1']

In [51]:
hours = np.arange(1, 25, 1)
n_warming_levels = len(warming_levels)
n_simulations = len(simulations)

# test = _create_multi_wl_single_sim_dataframe(
#         profile_data,
#         warming_levels,
#         simulations[0],
#         sim_label_func,
#         days_in_year,
#         hours,
#         hours_per_day,
#         hours_per_year
#     )


simulation = simulations[0]

In [ ]:
simulations

['sim1']

In [ ]:
sim_name = simulations[0] #sim_label_func(simulations, 0)
wl_names = [f"WL_{wl}" for wl in warming_levels]
# Create MultiIndex columns
col_tuples = [(sim, wl_name) for sim in simulations for wl_name in wl_names]
multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Sim", "Warming_Level"])


final_profile = pd.DataFrame(
    all_data,
    columns=multi_cols,
    index=np.arange(1, hours_per_year + 1, 1),
)

In [64]:
# Stack data
# all_data = _stack_profile_data(
#     profile_data=profile_data,
#     hours_per_day=hours_per_day,
#     wl_names=wl_names,
#     sim_names=[sim_name],
#     hour_first=True,
# )

sim_names = ['sim1-1']

all_data = []
for wl_name in wl_names:
    for sim_name in sim_names:
        key = (wl_name, sim_name)
        all_data.append(profile_data[key])

In [ ]:
all_data = np.column_stack(all_data)



In [67]:
final_profile

Sim                sim1          
Warming_Level    WL_1.5    WL_2.0
1              0.635826  0.880172
2              0.431987  0.137969
3              0.869548  0.593131
4              0.338154  0.943539
5              0.108490  0.093808
...                 ...       ...
8756           0.468230  0.939635
8757           0.670177  0.813707
8758           0.209208  0.345526
8759           0.450632  0.419856
8760           0.851353  0.678208

[8760 rows x 2 columns]

### Exploded final function

Alright, I see it now. Let's go from the top.

In [97]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5]
warming_levels_2 = [1.5, 2.0]
simulations = ["sim1"]
simulations_2 = ["sim1","sim2"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

simple_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels_2), len(time_delta), len(simulations))
multi_wl_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels_2,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations_2))
multi_sim_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations_2,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels_2), len(time_delta), len(simulations_2))
multi_both_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels_2,
        "time_delta": time_delta,
        "simulation": simulations_2,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)


In [98]:
def _stack_profile_data(
    profile_data: dict,
    wl_names: list,
    sim_names: list,
    hour_first: bool = True,
    three_level: bool = False,
) -> np.ndarray:
    """
    Stack profile data into a single array for DataFrame construction.

    Parameters
    ----------
    profile_data : dict
        Dictionary with (wl_name, sim_name) keys and profile arrays as values
    hours_per_day : int
        Number of hours per day (24)
    wl_names : list
        List of warming level names
    sim_names : list
        List of simulation names
    hour_first : bool, optional
        Whether hour should be the first level in iteration order
    three_level : bool, optional
        Whether this is a three-level MultiIndex (Hour, WL, Sim)

    Returns
    -------
    np.ndarray
        Stacked data array ready for DataFrame construction
    """
    all_data = []

    if three_level:
        # For three-level index: iterate hour -> wl -> sim
        for wl_name in wl_names:
            for sim_name in sim_names:
                key = (wl_name, sim_name)
                all_data.append(profile_data[key])
    elif hour_first:
        # For two-level with hour first
        for wl_name in wl_names:
            for sim_name in sim_names:
                key = (wl_name, sim_name)
                all_data.append(profile_data[key])

    return np.column_stack(all_data)

In [99]:
def _create_simple_dataframe(
    profile_data: dict,
    warming_level: float,
    simulation: any,
    sim_label_func: callable,
    hours_per_day: int,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create a simple DataFrame for single warming level and single simulation.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_level : float
        Single warming level value
    simulation : any
        Single simulation identifier
    sim_label_func : callable
        Function to get simulation label
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        Simple DataFrame with hour columns
    """

    wl_key = warming_level
    sim_key = sim_label_func(simulation, 0)

    # Stack data
    all_data = _stack_profile_data(
            profile_data=profile_data,
            wl_names=[f"WL_{wl_key}"],
            sim_names=[sim_key],
            hour_first=True,
        )

    return  pd.DataFrame(
            all_data,
            columns=[sim_key],
            index=np.arange(1, hours_per_year + 1, 1),
        )



In [100]:
def _create_single_wl_multi_sim_dataframe(
    profile_data: dict,
    warming_level: float,
    simulations: list,
    sim_label_func: callable,
    hours_per_day: int,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create DataFrame for single warming level with multiple simulations.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_level : float
        Single warming level value
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to get simulation labels
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        DataFrame with (Hour, Simulation) MultiIndex columns
    """
    wl = warming_level
    sim_names = [sim_label_func(sim, i) for i, sim in enumerate(simulations)]

    # Ensure simulation names are unique
    if len(sim_names) != len(set(sim_names)):
        print(
            "   ⚠️  Warning: Duplicate simulation names detected, adding uniqueness suffixes"
        )
        unique_sim_names = []
        name_counts = {}
        for name in sim_names:
            if name not in name_counts:
                name_counts[name] = 0
                unique_sim_names.append(name)
            else:
                name_counts[name] += 1
                unique_sim_names.append(f"{name}_v{name_counts[name]}")
        sim_names = unique_sim_names

    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        wl_names=[f"WL_{wl}"],
        sim_names=sim_names,
        hour_first=True,
    )
    
    return pd.DataFrame(
            all_data,
            columns=simulations,
            index=np.arange(1, hours_per_year + 1, 1),
        )




In [117]:
def _create_multi_wl_single_sim_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulation: any,
    sim_label_func: callable,
    hours_per_day: int,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create DataFrame for multiple warming levels with single simulation.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_levels : np.ndarray
        Array of warming level values
    simulation : any
        Single simulation identifier
    sim_label_func : callable
        Function to get simulation label
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        DataFrame with (Hour, Warming_Level) MultiIndex columns
    """

    sim_name = sim_label_func(simulation, 0)
    wl_names = [f"WL_{wl}" for wl in warming_levels]

    # Create MultiIndex columns
    col_tuples = [(wl_name, sim) for wl_name in wl_names for sim in simulations ]
    multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Warming_Level","Simulation"])


    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        wl_names=wl_names,
        sim_names=[sim_name],
        hour_first=True,
    )

    return pd.DataFrame(
        all_data,
        columns=multi_cols,
        index=np.arange(1, hours_per_year + 1, 1),
    )

In [118]:
def _create_multi_wl_multi_sim_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulations: list,
    sim_label_func: callable,
    hours_per_day: int,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create DataFrame for multiple warming levels and multiple simulations.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_levels : np.ndarray
        Array of warming level values
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to get simulation labels
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        DataFrame with (Hour, Warming_Level, Simulation) MultiIndex columns
    """
    wl_names = [f"WL_{wl}" for wl in warming_levels]
    sim_names = [sim_label_func(sim, i) for i, sim in enumerate(simulations)]

    # Ensure simulation names are unique
    if len(sim_names) != len(set(sim_names)):
        print(
            "   ⚠️  Warning: Duplicate simulation names detected, adding uniqueness suffixes"
        )
        unique_sim_names = []
        name_counts = {}
        for name in sim_names:
            if name not in name_counts:
                name_counts[name] = 0
                unique_sim_names.append(name)
            else:
                name_counts[name] += 1
                unique_sim_names.append(f"{name}_v{name_counts[name]}")
        sim_names = unique_sim_names

    # Create MultiIndex columns
    col_tuples = [
        (wl_name, sim_name)
        for wl_name in wl_names
        for sim_name in sim_names
    ]
    multi_cols = pd.MultiIndex.from_tuples(
        col_tuples, names=["Warming_Level", "Simulation"]
    )

    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        wl_names=wl_names,
        sim_names=sim_names,
        hour_first=True,
        three_level=True,
    )


    return pd.DataFrame(
        all_data,
        columns=multi_cols,
        index=np.arange(1, hours_per_year + 1, 1),
    )

In [119]:
def _construct_profile_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulations: list,
    sim_label_func: callable,
    hours_per_day: int,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Construct a DataFrame from profile data based on warming level and simulation dimensions.

    Parameters
    ----------
    profile_data : dict
        Dictionary with (warming_level, simulation) keys and profile arrays as values
    warming_levels : np.ndarray
        Array of warming level values
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to extract simulation labels
    days_in_year : int
        Number of days in the year (365 or 366)
    hours_per_day : int
        Number of hours per day (24)

    Returns
    -------
    pd.DataFrame
        Structured DataFrame with appropriate column structure based on dimensions
    """
    n_warming_levels = len(warming_levels)
    n_simulations = len(simulations)

    if n_warming_levels == 1 and n_simulations == 1:
        return _create_simple_dataframe(
            profile_data,
            warming_levels[0],
            simulations[0],
            sim_label_func,
            hours_per_day,
            hours_per_year
        )
    elif n_warming_levels == 1 and n_simulations > 1:
    
        return _create_single_wl_multi_sim_dataframe(
            profile_data,
            warming_levels[0],
            simulations,
            sim_label_func,
            hours_per_day,
            hours_per_year
        )
        
    elif n_warming_levels > 1 and n_simulations == 1:
        return _create_multi_wl_single_sim_dataframe(
            profile_data,
            warming_levels,
            simulations[0],
            sim_label_func,
            hours_per_day,
            hours_per_year
        )
    else:
        return _create_multi_wl_multi_sim_dataframe(
            profile_data,
            warming_levels,
            simulations,
            sim_label_func,
            hours_per_day,
            hours_per_year
        )


In [120]:
def test_compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time_delta)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
    )

    # Eagerly compute all dask data at once (one round-trip to scheduler)
    if hasattr(data.data, "chunks"):
        print("      📥 Loading data into memory...")
        from dask.diagnostics import ProgressBar

        with ProgressBar():
            data = data.compute()

    # Initialize storage for profiles
    profile_data = {}

    # Progress tracking
    total_combinations = len(warming_levels) * n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        for wl_idx, wl in enumerate(warming_levels):
            for sim_idx, sim in enumerate(simulations):
                # Get simulation label
                sim_label = _get_simulation_label(sim, sim_idx)

                # Select data for this warming level and simulation combination
                if has_simulation:
                    subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                else:
                    subset_data = data.isel(warming_level=wl_idx)

                # Vectorized quantile computation using numpy
                # Reshape raw values into (n_years, hours_per_year) then compute
                # the quantile across years for each hour-of-year position
                values = subset_data.values
                n_total = len(values)
                usable = (n_total // hours_per_year) * hours_per_year
                year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

                # Compute quantile targets for each of the 8760 hour positions
                quantile_targets = np.nanquantile(
                    year_hour_matrix, q, axis=0
                )  # shape: (8760,)

                # For each hour position, find the actual year whose value is
                # closest to the quantile (avoids interpolation)
                diffs = np.abs(
                    year_hour_matrix - quantile_targets[np.newaxis, :]
                )  # (n_years, 8760)
                closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)
                profile_1d = year_hour_matrix[
                    closest_year_idx, np.arange(hours_per_year)
                ]

                # Store the profile
                key = (f"WL_{wl}", sim_label)
                profile_data[key] = profile_1d

                pbar.update(1)

    df_profile = _construct_profile_dataframe(
        profile_data=profile_data,
        warming_levels=warming_levels,
        simulations=simulations,
        sim_label_func=_get_simulation_label,
        hours_per_day=hours_per_day,
        hours_per_year=hours_per_year
    )

    # #! -start
    # df_profile_reshaped = _construct_profile_dataframe(
    #     profile_data=profile_data_reshaped,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # df_profile_1d = _construct_profile_dataframe(
    #     profile_data=profile_data_1d,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -end

    # # Determine which formatting function to use based on the structure
    # _format_based_on_structure(df_profile)

    # #! -start
    # _format_based_on_structure(df_profile_reshaped)
    # _format_based_on_structure(df_profile_1d)
    # #! -end

    # # Prepare metadata dictionary
    # metadata = {
    #     "quantile": q,
    #     "method": "8760 analysis - actual data closest to quantile across 30 years",
    #     "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    # }

    # # Add original data attributes if available
    # if hasattr(data, "attrs"):
    #     if "units" in data.attrs:
    #         metadata["units"] = data.attrs["units"]
    #     if "extended_description" in data.attrs:
    #         metadata["extended_description"] = data.attrs["extended_description"]
    #     if "variable_id" in data.attrs:
    #         metadata["variable_name"] = data.attrs["variable_id"]
    #     elif hasattr(data, "name") and data.name:
    #         metadata["variable_name"] = data.name

    # # Set all metadata using the helper function
    # set_profile_metadata(df_profile, metadata)

    # #! -start
    # set_profile_metadata(df_profile_reshaped, metadata)
    # set_profile_metadata(df_profile_1d, metadata)
    # #! -end

    # print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    # print(
    #     f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    # )
    # if hasattr(data, "attrs") and "units" in data.attrs:
    #     print(f"         Units: {data.attrs['units']}")

    return df_profile #profile_data_reshaped, profile_data_1d

In [121]:
test_simple = test_compute_profile(simple_data,days_in_year=365, q=0.5)
test_multi_sim = test_compute_profile(multi_sim_data,days_in_year=365, q=0.5)
test_multi_both = test_compute_profile(multi_both_data,days_in_year=365, q=0.5)
test_multi_wl = test_compute_profile(multi_wl_data, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/1 [00:00<?, ?combo/s]

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 2 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 2 simulation(s)


      Computing profiles:   0%|          | 0/4 [00:00<?, ?combo/s]

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

In [122]:
test_multi_wl

Warming_Level,WL_1.5,WL_2.0
Simulation,sim1,sim1
1,0.884783,0.665231
2,0.769621,0.156413
3,0.244757,0.847318
4,0.007092,0.702253
5,0.824446,0.037726
...,...,...
8756,0.165638,0.522598
8757,0.475197,0.156599
8758,0.808972,0.280422


## Testing

In [ ]:
# Set up the Standard Year generator
profile_selections = {  
    ## Required variable and profile arguments
    "variable": "Air Temperature at 2m",
    "resolution": "3 km",
    "q": 0.5,
    "units": "degF",

    ## Required approach arguments, Options: "Warming Level", "Time"
    "approach": "Warming Level",
    # "warming_level": [warming_level], # GWL-option only
    # "centered_year": centered_year, # Time-based option only
    
    ## Required location arguments -- Uncomment your desired location option and modify
    "stations": ["Sacramento Executive Airport (KSAC)"], 
    # "latitude": (latitude - 0.02, latitude + 0.02), 
    # "longitude": (longitude - 0.02, longitude + 0.02),  
    # "cached_area": area_name, 

    ## Additional optional arguments -- Uncomment any desired options and modify
    # "no_delta": True, 
    # "warming_level_window": 10,
    # "time_profile_scenario": "SSP2-4.5,
    # "bias_adjusted_models": True,
}

# Generate the climate profile
profile = get_climate_profile(**profile_selections)

In [ ]:
# the function uses the previously defined profile selections to generate the output file name
export_profile_to_csv(profile, **profile_selections)